In [1]:
%load_ext autoreload
%autoreload 2

In [23]:
import pandas as pd
from preprocess import process_description, process_shop_category_name, process_vendor_name, format_category_name
import polars as pl
from sentence_transformers import SentenceTransformer
import torch
from sklearn.cluster import KMeans
import html
import re
import numpy as np

In [17]:
df = pd.read_csv("train.tsv", sep="\t")

In [18]:
# top_vendor_names = df["vendor_name"].lower().value_counts().head(11).index.to_list()
# top_vendor_names
top_vendor_names = [
    'нет бренда',
    'jiemiwl',
    'romiky',
    'jiemi',
    'джи чонг',
    'nrg',
    'xiaomi',
    'juxiangying',
    'heimerdinger',
    'linglingmaoyi',
    'muzimaoyi'
]

In [19]:
df = df.drop("vendor_code", axis=1)
df = process_description(df)
df = process_vendor_name(df, top_vendor_names)


In [6]:
model = SentenceTransformer("cointegrated/rubert-tiny2")

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 389.55it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
embeddings = model.encode(df["shop_category_name"].to_list(), convert_to_tensor=True)

In [8]:
embeddings_copy = embeddings.clone()
no_cat_name_mask = (df["shop_category_name"] == "-").to_numpy().astype(bool)
embeddings_copy[no_cat_name_mask] = 0

In [9]:
kmeans = KMeans(16)
clusters = kmeans.fit_predict(embeddings_copy.cpu().numpy())

In [27]:
top_category_names = [
    '',
    'женский',
    'мужской',
    'девочки',
    'автомагнитолы',
    'игрушки',
    'автоаксессуары',
    'аксессуарыдляприготовленияпищи',
    'инструментыдляремонтаистроительства',
    'электроустановочныеизделия',
    'декоративнаякосметика',
    'столоваяпосуда',
    'мальчики',
    'декориинтерьер',
    'измерительныйинструмент',
    'бытовоеосвещение',
    'тренажеры',
    'аксессуарыипринадлежностидлярыбалки',
    'запчастидлялегковыхавтомобилей',
    'посудадляприготовления',
    'мотоаксессуары'
]

In [30]:
df = process_shop_category_name(df, top_category_names, model, kmeans)

In [32]:
df['shop_category_name_cluster'].value_counts()

shop_category_name_cluster
7     3649
10    2440
0      884
6      469
15     326
1      285
5      263
13     185
2       72
12      21
14       6
8        5
Name: count, dtype: int64